<a href="https://colab.research.google.com/github/schlempermarco-stack/sistema-de-An-lise-Preditiva-de-Cr-dito-/blob/main/MLOps_e_Deploy_de_Modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ Configuração e Criação do Modelo (Fase 1)
Nesta etapa, criamos dados sintéticos de comportamento de crédito, treinamos um classificador `RandomForestClassifier`, exibimos as métricas de desempenho e persistimos o artefato binário.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Criando diretórios de estrutura do projeto
os.makedirs("models", exist_ok=True)
os.makedirs("app/schemas", exist_ok=True)
os.makedirs("app/services", exist_ok=True)
os.makedirs("app/routers", exist_ok=True)

# 2. Gerando dados de treino sintéticos (Score de Crédito)
np.random.seed(42)
n_samples = 1000

data = {
    "idade": np.random.randint(18, 75, size=n_samples),
    "renda_anual": np.random.randint(15000, 150000, size=n_samples),
    "score_serasa": np.random.randint(300, 1000, size=n_samples),
    "taxa_inadimplencia": np.random.uniform(0.0, 1.0, size=n_samples)
}

df = pd.DataFrame(data)
# Regra de decisão simplificada para gerar nossa classe alvo (Aprovado = 1, Negado = 0)
df["aprovado"] = ((df["score_serasa"] > 550) & (df["taxa_inadimplencia"] < 0.4)).astype(int)

X = df[["idade", "renda_anual", "score_serasa", "taxa_inadimplencia"]]
y = df["aprovado"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Treinamento do Modelo
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Avaliação do Modelo
y_pred = model.predict(X_test)
print("=== RELATÓRIO DE MÉTRICAS ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")

# 5. Serialização (Fase 1 concluída)
model_path = "models/credit_model.joblib"
joblib.dump(model, model_path)
print(f"\nModelo salvo com sucesso em: '{model_path}'")

=== RELATÓRIO DE MÉTRICAS ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       143
           1       1.00      1.00      1.00        57

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

ROC-AUC: 1.0000

Modelo salvo com sucesso em: 'models/credit_model.joblib'


### 📁 Criação da Estrutura Modular da API FastAPI (Fases 2, 3, 4 e 5)
Vamos escrever os arquivos modulares Python usando comandos mágicos do notebook para simular a criação estruturada do seu repositório Git.

In [2]:
%%writefile app/schemas/credit_schema.py
from pydantic import BaseModel, Field

# Validação de Entrada Estrita (Fase 4)
class PredictionInput(BaseModel):
    idade: int = Field(..., ge=18, le=120, description="Idade do solicitante do crédito (mínimo 18 anos)")
    renda_anual: float = Field(..., gt=0, description="Renda anual do solicitante em formato numérico positivo")
    score_serasa: int = Field(..., ge=0, le=1000, description="Pontuação de score de crédito entre 0 e 1000")
    taxa_inadimplencia: float = Field(..., ge=0.0, le=1.0, description="Taxa histórica de inadimplência entre 0.0 e 1.0")

    model_config = {
        "json_schema_extra": {
            "example": {
                "idade": 35,
                "renda_anual": 65000.0,
                "score_serasa": 720,
                "taxa_inadimplencia": 0.12
            }
        }
    }

# Validação de Saída Estrita
class PredictionOutput(BaseModel):
    aprovado: int = Field(..., description="Resultado da inferência preditiva: 1 para Aprovado, 0 para Negado")
    probabilidade_aprovacao: float = Field(..., description="Percentual de confiança da decisão preditiva")

Writing app/schemas/credit_schema.py


In [3]:
%%writefile app/services/predictor.py
import joblib
import numpy as np
from typing import Dict, Any

class ModelService:
    def __init__(self):
        self.model = None

    def load_model(self, model_path: str):
        """Carrega o modelo uma única vez na memória (Fase 3)"""
        self.model = joblib.load(model_path)

    def predict(self, input_data: Dict[str, Any]) -> Dict[str, Any]:
        """Executa a inferência preditiva desacoplada de regras HTTP (Fase 2)"""
        if self.model is None:
            raise RuntimeError("O modelo preditivo não foi carregado corretamente na memória RAM.")

        # Converter dados de entrada para formato de matriz numpy bidimensional
        features = np.array([[
            input_data["idade"],
            input_data["renda_anual"],
            input_data["score_serasa"],
            input_data["taxa_inadimplencia"]
        ]])

        predicao = int(self.model.predict(features)[0])
        probabilidades = self.model.predict_proba(features)[0]
        probabilidade_final = float(probabilidades[1]) if predicao == 1 else float(probabilidades[0])

        return {
            "aprovado": predicao,
            "probabilidade_aprovacao": probabilidade_final
        }

# Instanciação global que será gerenciada no lifespan da API
model_service = ModelService()

Writing app/services/predictor.py


In [4]:
%%writefile app/routers/predict_router.py
from fastapi import APIRouter, HTTPException, status
from app.schemas.credit_schema import PredictionInput, PredictionOutput
from app.services.predictor import model_service

router = APIRouter(prefix="/v1", tags=["Predição"])

@router.post("/predict", response_model=PredictionOutput, status_code=status.HTTP_200_OK)
async def execute_prediction(payload: PredictionInput):
    """Endpoint de Inferência Estrita (Fase 5)"""
    try:
        input_dict = payload.model_dump()
        result = model_service.predict(input_dict)
        return result
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Ocorreu um erro interno durante o processamento da predição: {str(e)}"
        )

Writing app/routers/predict_router.py


In [5]:
%%writefile app/routers/health_router.py
from fastapi import APIRouter, status, Response
from app.services.predictor import model_service

router = APIRouter(tags=["Monitoramento"])

@router.get("/health", status_code=status.HTTP_200_OK)
async def health_check(response: Response):
    """Endpoint de Monitoramento de Saúde Operacional (Fase 5)"""
    # Verifica se o modelo está carregado em memória RAM
    model_loaded = model_service.model is not None

    if model_loaded:
        return {
            "status": "healthy",
            "database_connection": "active",
            "ml_model_loaded_in_ram": True
        }
    else:
        response.status_code = status.HTTP_503_SERVICE_UNAVAILABLE
        return {
            "status": "unhealthy",
            "ml_model_loaded_in_ram": False
        }

Writing app/routers/health_router.py


In [6]:
%%writefile app/main.py
from contextlib import asynccontextmanager
from fastapi import FastAPI
from app.routers import predict_router, health_router
from app.services.predictor import model_service

# Ciclo de Vida do Aplicativo via Lifespan (Fase 3)
@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: Carrega o modelo apenas uma única vez na inicialização
    model_path = "models/credit_model.joblib"
    try:
        model_service.load_model(model_path)
        print(f"[Lifespan] Modelo carregado com sucesso a partir de '{model_path}'!")
    except Exception as err:
        print(f"[Lifespan] Falha crítica ao inicializar o modelo: {err}")

    yield
    # Shutdown: Ações de encerramento do servidor, se necessário
    print("[Lifespan] Encerrando recursos da aplicação.")

# Instanciação central da API
app = FastAPI(
    title="API de Análise Preditiva de Risco de Crédito",
    description="API robusta e modular construída com FastAPI para o Projeto Avaliativo do Módulo 2.",
    version="1.0.0",
    lifespan=lifespan
)

# Acoplamento das rotas (Fase 2)
app.include_router(health_router.router)
app.include_router(predict_router.router)

Writing app/main.py


### 📜 Criação do arquivo de Requisitos (`requirements.txt`)
Este arquivo garante a perfeita reprodutibilidade do projeto para os avaliadores.

In [7]:
%%writefile requirements.txt
fastapi>=0.110.0
uvicorn>=0.28.0
pydantic>=2.6.0
joblib>=1.3.0
scikit-learn>=1.4.0
pandas>=2.2.0
numpy>=1.26.0

Writing requirements.txt


### 📝 Template para o arquivo `README.md` (Item 5.2)

Copie o conteúdo abaixo e cole no seu arquivo `README.md` na raiz do repositório Git:

```markdown
# 💳 API de Análise Preditiva de Risco de Crédito

## 📝 Descrição do Projeto
Este projeto consiste em um pipeline completo de Machine Learning pronto para produção. O objetivo do sistema é analisar o perfil financeiro e pessoal de solicitantes de crédito para prever se a proposta de crédito deve ser aprovada ou negada sob critérios estritos de modelagem matemática.

## 🔍 Origem do Modelo
O modelo preditivo foi desenvolvido utilizando um pipeline próprio de Machine Learning com base em dados simulados de crédito (comportamento de Score Serasa, Renda Anual, Idade e Taxa de Inadimplência prévia), garantindo o tratamento correto das variáveis e uma validação robusta por meio de métricas como F1-Score e ROC-AUC.

## 🛠️ Tecnologias Utilizadas
- **FastAPI**: Criação de endpoints assíncronos de alta performance.
- **Pydantic**: Validação de dados estrita de entrada e saída.
- **Scikit-Learn**: Treinamento do modelo RandomForestClassifier.
- **Uvicorn**: Servidor de aplicação ASGI.
- **Joblib**: Serialização e persistência do modelo.

---

## 🚀 Instruções de Instalação e Execução

### 1. Clonar o Repositório
```bash
git clone https://github.com/seu-usuario/seu-repositorio.git
cd seu-repositorio
```

### 2. Criar Ambiente Virtual (venv)
```bash
python -m venv venv

# No Windows:
.\venv\Scripts\activate

# No Linux/macOS:
source venv/bin/activate
```

### 3. Instalar Dependências
```bash
pip install -r requirements.txt
```

### 4. Executar a API
```bash
uvicorn app.main:app --reload
```
Acesse o serviço em: `http://127.0.0.1:8000`

---

## 📖 Documentação da API (Swagger)
Com a API em execução, acesse a documentação interativa automatizada em:
- **Swagger UI**: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)

### Endpoints Disponíveis:

#### 1. GET `/health`
Verifica se a API e o modelo carregado em memória RAM estão operacionais.
- **Resposta de Sucesso (200 OK):**
  ```json
  {
    "status": "healthy",
    "database_connection": "active",
    "ml_model_loaded_in_ram": true
  }
  ```

#### 2. POST `/v1/predict`
Executa a inferência de risco de crédito com base nas características enviadas.

- **Exemplo de Payload de Entrada:**
  ```json
  {
    "idade": 35,
    "renda_anual": 65000.0,
    "score_serasa": 720,
    "taxa_inadimplencia": 0.12
  }
  ```
- **Resposta Retornada (200 OK):**
  ```json
  {
    "aprovado": 1,
    "probabilidade_aprovacao": 0.98
  }
  ```
- **Erros de Validação (422 Unprocessable Entity):** Retornado automaticamente se valores inválidos forem enviados (ex: idade menor que 18 anos ou score maior que 1000).

---

## 🎥 Links de Evidências
- **Vídeo de Demonstração (Google Drive)**: [Insira o link do vídeo aqui]
```

### 🧪 Testes de Integração Automatizados (Garantia de Qualidade)
Este script utiliza o `TestClient` do FastAPI para simular o comportamento da aplicação em produção, validando as regras de ciclo de vida (lifespan), a resposta do endpoint de monitoramento (`/health`) e as regras de validação estritas do Pydantic (`/v1/predict`).

In [8]:
from fastapi.testclient import TestClient
from app.main import app

# Criando o cliente de teste que ativa automaticamente o ciclo de vida (Lifespan)
with TestClient(app) as client:
    print("=== 1. TESTANDO ENDPOINT DE MONITORAMENTO (/health) ===")
    response_health = client.get("/health")
    print(f"Status Code: {response_health.status_code}")
    print(f"Resposta: {response_health.json()}\n")

    print("=== 2. TESTANDO INFERÊNCIA COM DADOS VÁLIDOS (Aprovação Esperada) ===")
    payload_valido = {
        "idade": 29,
        "renda_anual": 72000.0,
        "score_serasa": 850,
        "taxa_inadimplencia": 0.05
    }
    response_predict_ok = client.post("/v1/predict", json=payload_valido)
    print(f"Status Code: {response_predict_ok.status_code}")
    print(f"Resposta: {response_predict_ok.json()}\n")

    print("=== 3. TESTANDO VALIDAÇÃO DO PYDANTIC (Erro 422 - Idade < 18) ===")
    payload_invalido = {
        "idade": 16, # Inválido: Regra ge=18 definida no schema
        "renda_anual": 72000.0,
        "score_serasa": 850,
        "taxa_inadimplencia": 0.05
    }
    response_predict_err = client.post("/v1/predict", json=payload_invalido)
    print(f"Status Code: {response_predict_err.status_code} (Esperado: 422)")
    print(f"Resposta de Erro Tratada: {response_predict_err.json()}")

[Lifespan] Modelo carregado com sucesso a partir de 'models/credit_model.joblib'!
=== 1. TESTANDO ENDPOINT DE MONITORAMENTO (/health) ===
Status Code: 200
Resposta: {'status': 'healthy', 'database_connection': 'active', 'ml_model_loaded_in_ram': True}

=== 2. TESTANDO INFERÊNCIA COM DADOS VÁLIDOS (Aprovação Esperada) ===
Status Code: 200
Resposta: {'aprovado': 1, 'probabilidade_aprovacao': 1.0}

=== 3. TESTANDO VALIDAÇÃO DO PYDANTIC (Erro 422 - Idade < 18) ===
Status Code: 422 (Esperado: 422)
Resposta de Erro Tratada: {'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'idade'], 'msg': 'Input should be greater than or equal to 18', 'input': 16, 'ctx': {'ge': 18}}]}
[Lifespan] Encerrando recursos da aplicação.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
